# 01 — Runtime and exact recovery

**Status: UNEXECUTED FIRST DRAFT.** Establish interruption-safe optimization before any long data, training, evaluation or HPO work.

- **Inputs:** notebook 00 completed, AHR v2 XL kernel, all 16 notebook source files.
- **Outputs:** common runtime definitions, hashed checkpoint manifests, `reports/recovery_gate.json` after an actual passing check.
- **Expected runtime:** under two minutes for the CPU diagnostic; no AHR learning happens here.
- **Checkpoint location:** `checkpoints/runtime-recovery-<timestamp>/`; every experiment later uses `checkpoints/<run_id>/`.
- **Resume:** rerun this notebook for a new diagnostic receipt. For real runs, keep the same run ID and configuration and rerun the stage's **Run or resume** cell. It restores the most recent valid checkpoint; it never silently starts over when checkpoint files exist.

Checkpoints contain model, optimizer, scheduler, scaler, Python/NumPy/torch/CUDA RNG, data/sampler position, source/config identity, metrics and caller-provided replay/episode state. Cadence is **600 seconds or 200 optimizer steps**, plus every stage/epoch boundary. The manifest retains the latest two and best checkpoint. Exact optimization continuation assumes an unchanged runtime, deterministic operators and identical source/data. The external ARC service has separate recovery limits; a local checkpoint cannot snapshot hidden server state.

## 1. Load only shared definitions

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())


## 2. Determinism, sampler and exclusive run ownership

In [ ]:
import contextlib, dataclasses, random, re, socket, uuid
import numpy as np
import torch

def seed_everything(seed):
    os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

def capture_rng():
    return {'python': random.getstate(), 'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_initialized() else None}

def restore_rng(state):
    random.setstate(state['python']); np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'].cpu())
    if state.get('cuda') is not None:
        if len(state['cuda']) != torch.cuda.device_count():
            raise RuntimeError('CUDA device count changed; exact RNG continuation is unavailable.')
        torch.cuda.set_rng_state_all([s.cpu() for s in state['cuda']])

def runtime_fingerprint():
    return {'python': platform.python_version(), 'torch': torch.__version__,
            'numpy': np.__version__, 'cuda_build': torch.version.cuda,
            'cuda_device_count': torch.cuda.device_count(),
            'deterministic_algorithms': torch.are_deterministic_algorithms_enabled()}

class StatefulSampler:
    """Deterministic without-replacement epochs, including batches crossing a boundary."""
    def __init__(self, size, seed):
        if int(size) < 1: raise ValueError('Cannot sample an empty dataset')
        self.size = int(size); self.rng = np.random.default_rng(seed)
        self.epoch = 0; self.position = 0
        self.order = self.rng.permutation(self.size)
    def next_indices(self, batch_size):
        if int(batch_size) < 1: raise ValueError('batch_size must be positive')
        result = []
        while len(result) < int(batch_size):
            if self.position == self.size:
                self.order = self.rng.permutation(self.size)
                self.position = 0; self.epoch += 1
            n = min(int(batch_size) - len(result), self.size - self.position)
            result.extend(self.order[self.position:self.position + n].tolist())
            self.position += n
        return result
    def state_dict(self):
        return {'size': self.size, 'epoch': self.epoch, 'position': self.position,
                'order': self.order.tolist(), 'rng': copy.deepcopy(self.rng.bit_generator.state)}
    def load_state_dict(self, state):
        if int(state['size']) != self.size: raise ValueError('Sampler dataset size changed')
        order = np.asarray(state['order'], dtype=np.int64)
        if sorted(order.tolist()) != list(range(self.size)): raise ValueError('Invalid sampler permutation')
        if not 0 <= int(state['position']) <= self.size: raise ValueError('Invalid sampler position')
        self.epoch = int(state['epoch']); self.position = int(state['position']); self.order = order
        self.rng.bit_generator.state = copy.deepcopy(state['rng'])

def append_metrics(path, row):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a') as stream:
        stream.write(json.dumps(row, sort_keys=True, allow_nan=False) + '\n')
        stream.flush(); os.fsync(stream.fileno())

class RunLock:
    """OS advisory lock; process death releases it without deleting another writer's lock."""
    def __init__(self, root, run_id):
        if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{0,127}', str(run_id)):
            raise ValueError('Use a simple run_id without paths')
        self.path = Path(root) / 'checkpoints' / str(run_id) / '.run.lock'
        self.stream = None
    def __enter__(self):
        import fcntl
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.stream = self.path.open('a+')
        try: fcntl.flock(self.stream.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError:
            self.stream.close(); self.stream = None
            raise RuntimeError(f'Another process already owns this run: {self.path.parent.name}')
        self.stream.seek(0); self.stream.truncate()
        self.stream.write(json.dumps({'pid': os.getpid(), 'host': socket.gethostname(), 'started': time.time()}))
        self.stream.flush()
        return self
    def __exit__(self, exc_type, exc, tb):
        import fcntl
        if self.stream is not None:
            fcntl.flock(self.stream.fileno(), fcntl.LOCK_UN)
            self.stream.close(); self.stream = None


## 3. Atomic checkpoint commits and validated recovery

In [ ]:
class CheckpointManager:
    """Own trusted torch checkpoints only. Never load downloaded/untrusted pickle payloads."""
    def __init__(self, root, run_id, config, source_hashes=None):
        if not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_.-]{0,127}', str(run_id)):
            raise ValueError('Unsafe run_id')
        self.root = Path(root); self.run_id = str(run_id)
        self.directory = self.root / 'checkpoints' / self.run_id
        self.directory.mkdir(parents=True, exist_ok=True)
        self.manifest_path = self.directory / 'manifest.json'
        self.config = copy.deepcopy(config)
        self.config_hash = canonical_hash(self.config)
        self.source_hashes = source_manifest(self.root) if source_hashes is None else dict(source_hashes)
        self.last_saved_step = 0; self.last_saved_time = time.monotonic()
        cadence = config.get('train', {})
        self.max_seconds = min(float(cadence.get('checkpoint_seconds', 600)), 900)
        self.max_steps = min(int(cadence.get('checkpoint_steps', 200)), 250)
        if self.max_seconds <= 0 or self.max_steps <= 0: raise ValueError('Invalid checkpoint cadence')
    def due(self, step):
        return (int(step) - self.last_saved_step >= self.max_steps or
                time.monotonic() - self.last_saved_time >= self.max_seconds)
    def _identity_check(self, record):
        if record.get('config_hash') != self.config_hash:
            raise RuntimeError('Configuration/data identity changed. Use a NEW run_id for a new experiment.')
        if record.get('source_hashes') != self.source_hashes:
            raise RuntimeError('Notebook source changed. Exact continuation requires the original source; use a new run_id for a deliberate warm start.')
    def _records(self):
        records, damaged_metadata = {}, []
        def identity_shape(value):
            return (isinstance(value, dict) and isinstance(value.get('config_hash'), str)
                    and isinstance(value.get('source_hashes'), dict))
        def record_shape(value):
            return (identity_shape(value) and isinstance(value.get('file'), str)
                    and isinstance(value.get('sha256'), str)
                    and type(value.get('step')) is int and type(value.get('saved_ns')) is int)
        if self.manifest_path.exists():
            try:
                manifest = json.loads(self.manifest_path.read_text())
                if not identity_shape(manifest) or not isinstance(manifest.get('checkpoints'), list):
                    raise ValueError('Malformed checkpoint manifest')
            except (OSError, ValueError):
                damaged_metadata.append(self.manifest_path.name)
            else:
                self._identity_check(manifest)
                items = manifest['checkpoints'] + ([manifest['best']] if manifest.get('best') else [])
                for item in items:
                    if record_shape(item):
                        self._identity_check(item)
                        records[item['file']] = item
                    else:
                        damaged_metadata.append(self.manifest_path.name + ':record')
        # Independent commit markers recover a missing/damaged manifest or interrupted replacement.
        for path in self.directory.glob('step-*.commit.json'):
            try:
                item = json.loads(path.read_text())
                if not record_shape(item): raise ValueError('Malformed checkpoint record')
            except (OSError, ValueError):
                damaged_metadata.append(path.name)
                continue
            self._identity_check(item)
            records[item['file']] = item
        if not records and (damaged_metadata or list(self.directory.glob('step-*.pt'))):
            raise RuntimeError('Checkpoint artifacts exist without valid metadata. Refusing silent restart; inspect this run.')
        if damaged_metadata and records:
            print('Recovering from independent checkpoint records; unreadable metadata:', damaged_metadata)
        return sorted(records.values(), key=lambda x: (x['step'], x['saved_ns']), reverse=True)
    def _valid_record(self, item):
        name = item.get('file', '')
        if Path(name).name != name or not name.endswith('.pt'): return False
        path = self.directory / name
        return path.is_file() and sha256_file(path) == item.get('sha256')
    def save(self, model, optimizer=None, scheduler=None, scaler=None, step=0,
             metrics=None, extra=None, is_best=False):
        previous = [record for record in self._records() if self._valid_record(record)]
        stamp = time.time_ns()
        name = f'step-{int(step):09d}-{stamp}.pt'
        target = self.directory / name
        temporary = self.directory / (name + '.tmp-' + uuid.uuid4().hex)
        payload = {'format_version': 1, 'model': model.state_dict(),
                   'optimizer': optimizer.state_dict() if optimizer is not None else None,
                   'scheduler': scheduler.state_dict() if scheduler is not None else None,
                   'scaler': scaler.state_dict() if scaler is not None else None,
                   'rng': capture_rng(), 'step': int(step), 'metrics': metrics or {},
                   'extra': extra or {}, 'config': self.config, 'config_hash': self.config_hash,
                   'source_hashes': self.source_hashes, 'runtime': runtime_fingerprint()}
        try:
            with temporary.open('wb') as stream:
                torch.save(payload, stream); stream.flush(); os.fsync(stream.fileno())
            os.replace(temporary, target)
            item = {'file': name, 'sha256': sha256_file(target), 'step': int(step),
                    'saved_ns': stamp, 'is_best': bool(is_best),
                    'config_hash': self.config_hash, 'source_hashes': self.source_hashes}
            atomic_json(self.directory / (name[:-3] + '.commit.json'), item)
            latest = [item] + previous
            prior_best = next((x for x in previous if x.get('is_best') and self._valid_record(x)), None)
            best = item if is_best else prior_best
            retained = latest[:2]
            if best and best['file'] not in {x['file'] for x in retained}: retained.append(best)
            atomic_json(self.manifest_path, {
                'format_version': 1, 'run_id': self.run_id, 'latest': item,
                'best': best, 'checkpoints': retained, 'resume_cell': 'Run or resume',
                'config_hash': self.config_hash, 'source_hashes': self.source_hashes})
            keep = {x['file'] for x in retained}
            for old in previous:
                if old['file'] not in keep:
                    (self.directory / old['file']).unlink(missing_ok=True)
                    (self.directory / (old['file'][:-3] + '.commit.json')).unlink(missing_ok=True)
            self.last_saved_step = int(step); self.last_saved_time = time.monotonic()
            return target
        finally:
            temporary.unlink(missing_ok=True)
    def latest_payload(self, map_location='cpu'):
        records = self._records()
        if not records: return None
        failures = []
        for item in records:
            if not self._valid_record(item):
                failures.append(item['file']); continue
            try:
                payload = torch.load(self.directory / item['file'], map_location=map_location, weights_only=False)
            except (OSError, RuntimeError, EOFError):
                failures.append(item['file']); continue
            self._identity_check(payload)
            for key in ('python', 'torch', 'numpy', 'cuda_build'):
                if payload['runtime'].get(key) != runtime_fingerprint().get(key):
                    raise RuntimeError(f'Runtime {key} changed; exact continuation has not been validated.')
            payload['checkpoint_path'] = str(self.directory / item['file'])
            payload['checkpoint_sha256'] = item['sha256']
            if failures: print('Recovered from earlier valid checkpoint; rejected:', failures)
            return payload
        raise RuntimeError(f'No valid resumable checkpoint in {self.directory}; refusing silent restart.')
    def load_latest(self, model, optimizer=None, scheduler=None, scaler=None, map_location='cpu'):
        payload = self.latest_payload(map_location)
        if payload is None: return None
        model.load_state_dict(payload['model'], strict=True)
        for name, obj in (('optimizer', optimizer), ('scheduler', scheduler), ('scaler', scaler)):
            if obj is not None:
                if payload.get(name) is None: raise RuntimeError(f'Checkpoint lacks requested {name} state')
                obj.load_state_dict(payload[name])
        restore_rng(payload['rng'])
        self.last_saved_step = int(payload['step']); self.last_saved_time = time.monotonic()
        print(f"Resumed {self.run_id} at step {payload['step']}; metrics={payload['metrics']}")
        return payload


## 4. Exact continuation diagnostic and gate semantics

In [ ]:
def _nested_equal(a, b):
    if torch.is_tensor(a): return torch.is_tensor(b) and torch.equal(a.cpu(), b.cpu())
    if isinstance(a, np.ndarray): return isinstance(b, np.ndarray) and np.array_equal(a, b)
    if isinstance(a, dict): return isinstance(b, dict) and a.keys() == b.keys() and all(_nested_equal(a[k], b[k]) for k in a)
    if isinstance(a, (list, tuple)): return type(a) is type(b) and len(a) == len(b) and all(_nested_equal(x, y) for x, y in zip(a, b))
    return a == b

def validate_recovery(root=None):
    root = ensure_layout(root or ROOT)
    seed_everything(1024)
    run_id = 'runtime-recovery-' + str(time.time_ns())
    diagnostic_config = {'fixture': 'cpu-adamw-exact-continuation', 'version': 1}
    model = torch.nn.Sequential(torch.nn.Linear(4, 8), torch.nn.GELU(), torch.nn.Linear(8, 1))
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.01)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
    sampler = StatefulSampler(19, 1024)
    manager = CheckpointManager(root, run_id, diagnostic_config)
    def advance():
        indices = sampler.next_indices(5)
        x = torch.randn(5, 4) + torch.tensor(indices).float().unsqueeze(1) / 19
        y = x.sum(-1, keepdim=True)
        optimizer.zero_grad(set_to_none=True)
        loss = torch.nn.functional.mse_loss(model(x), y)
        loss.backward(); optimizer.step(); scheduler.step()
        return float(loss.detach())
    with RunLock(root, run_id):
        for _ in range(3): saved_loss = advance()
        manager.save(model, optimizer, scheduler, step=3, metrics={'fixture_loss': saved_loss},
                     extra={'sampler': sampler.state_dict(), 'event_state': {'observed': [1, 2], 'imagined': []}})
        expected_loss = advance()
        expected_weights = copy.deepcopy(model.state_dict())
        expected_optimizer = copy.deepcopy(optimizer.state_dict())
        expected_scheduler = copy.deepcopy(scheduler.state_dict())
        expected_sampler = sampler.state_dict()
        expected_rng = capture_rng()
        payload = manager.load_latest(model, optimizer, scheduler)
        sampler.load_state_dict(payload['extra']['sampler'])
        actual_loss = advance()
        checks = {
            'loss': expected_loss == actual_loss,
            'parameters': _nested_equal(expected_weights, model.state_dict()),
            'optimizer': _nested_equal(expected_optimizer, optimizer.state_dict()),
            'scheduler': _nested_equal(expected_scheduler, scheduler.state_dict()),
            'sampler': _nested_equal(expected_sampler, sampler.state_dict()),
            'rng': _nested_equal(expected_rng, capture_rng()),
            'extra_state': payload['extra']['event_state'] == {'observed': [1, 2], 'imagined': []},
        }
        result = {'passed': all(checks.values()), 'checks': checks, 'run_id': run_id,
                  'source_hashes': source_manifest(root), 'runtime': runtime_fingerprint(),
                  'created_at': time.time(), 'diagnostic_only': True}
        atomic_json(root / 'reports' / 'recovery_gate.json', result)
        if not result['passed']: raise RuntimeError(f'Exact continuation failed: {checks}')
        manager.save(model, optimizer, scheduler, step=4, metrics={'fixture_loss': actual_loss},
                     extra={'sampler': sampler.state_dict()}, is_best=True)
    return result

def require_recovery_gate(root=None):
    root = Path(root or ROOT)
    path = root / 'reports' / 'recovery_gate.json'
    if not path.exists(): raise RuntimeError('Run notebook 01 recovery validation before this stage.')
    gate = json.loads(path.read_text())
    if gate.get('passed') is not True or gate.get('source_hashes') != source_manifest(root):
        raise RuntimeError('Recovery gate missing, failed or stale after a code edit. Re-run notebook 01.')
    current = runtime_fingerprint()
    for key in ('python', 'torch', 'numpy', 'cuda_build'):
        if gate['runtime'].get(key) != current.get(key):
            raise RuntimeError(f'Recovery runtime changed ({key}); rerun notebook 01.')
    return gate


## 5. Run the diagnostic

A tiny CPU network is a **runtime fixture only**. It is not the compact AHR model and produces no benchmark evidence. This test compares the next loss, every parameter, optimizer/scheduler state, sampler, RNG and caller state after an interruption. It writes PASS only after these comparisons succeed. Full XL continuation is checked again in notebook 09.

In [ ]:
recovery_result = validate_recovery(ROOT)
print(json.dumps(recovery_result, indent=2))


## 6. Resume contract for later notebooks

If a run is interrupted, select the same kernel, restore the same configuration and source version, and Run All on that stage. The resume cell prints the recovered step and metrics before continuing. Stop requests and Python interrupts save at a consistent optimizer boundary; a killed process resumes the last committed checkpoint and can lose at most one checkpoint interval of work.

Configuration or dataset changes require a **new run ID**, with an explicit warm-start checkpoint if desired. Never disable the identity checks to force an incompatible resume. A local volume survives instance recreation but remains tied to its host; keep separate off-host copies of valuable best/final checkpoints before any destructive operation.

Next: notebook **02_arc_environment.ipynb**. No data collection or training is triggered here.